In [ ]:
import os
import os.path as op

import importlib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpecFromSubplotSpec

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.signal import argrelextrema
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression

from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
sys.path.append("../..")
path_to_data = op.join("../..", "data")

from bimodularity import dgsp
from bimodularity import bimod_plots as plot
from bimodularity import palettes
from bimodularity import data_load as dload
from bimodularity import bundle as b_utils

In [ ]:
cmaps = plot.get_all_cmaps()
print(cmaps.keys())

In [ ]:
path_to_brain_data = op.join("../../data", "brain")
path_to_bundles = op.join(path_to_brain_data, "SCIL")

path_to_derivatives = op.join("../../data", "brain", "derivatives")

path_to_sc = op.join(path_to_derivatives, "structural_connectome")
path_to_ec = op.join(path_to_derivatives, "atlas_correspondence", "rDCM")
path_to_lobe = op.join(path_to_derivatives, "atlas_correspondence", "Lobes")
path_to_consensus = op.join(path_to_derivatives, "consensus_clustering")

fast = False

In [ ]:
all_scales = [1, 2, 3, 4]

use_delay = False
k_threshold = 0
b_thresh = 0
slines_theshold = 5

struct_type = ""

scale = 2

k_matrix, labels, node_centers = dload.load_bundle_graph(path_to_data=path_to_sc,
                                                         data_suffix="Laus2018_",
                                                         scale=scale,
                                                         b_prob_threshold=0,
                                                         slines_theshold=slines_theshold,
                                                         log_slines=False,
                                                         normalize_slines=False,
                                                         verbose=True)

labels, node_centers = b_utils.fix_thalamus(labels, pos=node_centers)
order_by_lobe, lobe_sizes, lobe_labels, lobe_df = dload.get_lobe_info(scale, labels, path_to_lobe=path_to_lobe)

# labels, k_matrix = b_utils.fix_thalamus(labels, matrix=k_matrix)
# k_matrix[k_matrix < b_thresh] = 0

# ec_mat = dload.get_ec_data(scale=scale, path_to_ec=path_to_ec, remove_neg=True, fix_thal=True, verbose=True)

# gamma = 1
# ec_mat = ec_mat ** gamma
# dir_ratio = np.divide(ec_mat, (ec_mat + ec_mat.T),
#                       where=(ec_mat + ec_mat.T) > 0,
#                       out=np.zeros_like(ec_mat))
# graph = (k_matrix > 0) * dir_ratio

# U, S, Vh = dgsp.sorted_SVD(dgsp.modularity_matrix(graph))
# V = Vh.T

# n_nodes = len(S)

# fig, axes = plt.subplots(ncols=3, figsize=(20, 5))
# axes[0].imshow((k_matrix > 0)[order_by_lobe][:, order_by_lobe], cmap=cmaps["div_rb"], interpolation="none", vmax=2)
# axes[0].set_xticks([])
# axes[0].set_yticks([])
# axes[1].imshow(dir_ratio[order_by_lobe][:, order_by_lobe], cmap=cmaps["div_rb"], interpolation="none")
# axes[1].set_xticks([])
# axes[1].set_yticks([])

# axes[2].imshow(graph[order_by_lobe][:, order_by_lobe], cmap=cmaps["div_rb"], interpolation="none")
# fig, axes[2], cbar = plot.add_cbar(fig, axes[2])
# cbar.set_ticks([0, 0.5, 1], labels=["0", "Bidirectional", "1"], rotation=90, ha="left", va="center", fontsize=14)
# plot.plot_lobe_lines(axes[2], lobe_sizes, lobe_labels, x_hemi=True, no_insula=True, grid_color="w")

## Consensus Clustering

In [ ]:
scale = 2
n_vec_max = 20
gamma = 1

nosquared = False

n_trials = 50
n_init = 50
dthresh = 0
selected_sort = 0

upsample = 3

all_n_kmeans = np.arange(10, 80, 1)

bm_fname = "_".join([
    f"brain_consensus-EC",
    f"scale{scale}",
    f"nvec{n_vec_max}" + "-NoSquared"*nosquared,
    f"trials{n_trials}",
    f"ninit{n_init}",
    f"kmeans{all_n_kmeans[0]}-{all_n_kmeans[-1]}",
    f"slines_thresh{slines_theshold}",
    f"ustruct{struct_type}"*(struct_type != ""),
    f"dthresh{dthresh}"*(dthresh != 0),
    f"gamma{gamma}"*(gamma != 1),
    f"selected{selected_sort}"*(selected_sort != 0)
    ])
bm_fname += ".pkl"

if op.isfile(op.join(path_to_consensus, bm_fname)):
    data = dload.load(op.join(path_to_consensus, bm_fname))
    graph = data["graph"]
    all_n_kmeans = data["all_n_kmeans"]
    cons_lab = data["cons_lab"]
    avg_cons = data["avg_cons"]
    edge_to_bundle_mat = data["edge_to_bundle_mat"]
else:
    print("No benchmark file found - Please, run the benchmarking notebook!")
    print(f"\t`{bm_fname}`")

fig_suffix = f"gamma{gamma}_nvec{n_vec_max}" + "-NoSquared"*nosquared
path_to_figures = op.join("./figures", f"scale{scale}", fig_suffix)

os.makedirs(path_to_figures, exist_ok=True)

In [ ]:
centroid_dir = op.join(
    path_to_brain_data, "BundleAtlas", "centroids",
    f"scale{scale}", f"group_centroids_scale{scale}"
)

a_bundles_labels = pd.read_csv(op.join(path_to_bundles, "bundle_names.csv"))
a_bundles_labels = a_bundles_labels.rename(columns={"Unnamed: 0": "BundleNum"})

In [ ]:
if "edge_to_bundle_Bmdf" in data:
    edge_to_bundle_dist = data["edge_to_bundle_Bmdf"]
else:
    edge_to_bundle_dist = b_utils.compute_edge_to_bundle_distance(
        graph, labels=labels, scale=scale,
        path_to_edge_centroids=centroid_dir,
        path_to_atlas_centroids=path_to_bundles, a_bundles_labels=a_bundles_labels
    )

    data.update({"edge_to_bundle_Bmdf": edge_to_bundle_dist})
    dload.save(op.join(path_to_consensus, bm_fname), data)

## Hierarchical Consensus Clustering

### Preparation Code

In [ ]:
hierarchical_matrix = 1 - avg_cons
c_mat = 1 - hierarchical_matrix

dist_condensed = squareform(hierarchical_matrix)
Z = linkage(dist_condensed, method='average', metric='precomputed')
distances = np.flip(Z[:, 2])
diff_dist = -np.diff(distances)

In [ ]:
n_samples = len(hierarchical_matrix)

# Number of clusters K decreases from n_samples to 1
n_clusters = np.arange(1, n_samples)

nshow = 100
p_idx, q_idx = np.triu_indices(n_samples, k=1)

all_ratios = np.zeros(nshow)
all_self = np.zeros(nshow)
all_bi = np.zeros(nshow)

if not fast:
    for i, k in enumerate(n_clusters[:nshow]):
        labs = fcluster(Z, t=k, criterion='maxclust')

        pair_same = (labs[p_idx] == labs[q_idx])
        within_val = 1 - dist_condensed[pair_same]
        between_val = 1 - dist_condensed[~pair_same]

        within_mean = within_val[within_val > 0].mean()
        if len(between_val[between_val > 0]) == 0:
            all_ratios[i] = np.nan
            continue
        else:
            between_mean = between_val[between_val > 0].mean()

        all_ratios[i] = within_mean / between_mean

        e_labels = labs
        e_mat = np.zeros_like(graph, dtype=int)
        e_mat[graph > 0] = e_labels

        send, rec = dgsp.get_node_clusters(e_labels, e_mat)
        row_ind, col_ind = dgsp.get_conjugates_matching(send, rec)

        self = np.where(row_ind == col_ind)[0]
        all_self[i] = len(self)

all_ratios = np.nan_to_num(all_ratios, nan=np.nanmax(all_ratios))

best_k = np.argmax(diff_dist)+1
print(f"Best k is at max diff: {best_k} with distance jump {diff_dist[best_k-1]:.4f}")

# Plot branching distance vs number of clusters
fig, axes = plt.subplots(ncols=2, figsize=(12, 5))
axes[0].plot(n_clusters[:nshow], distances[:nshow], marker='o', zorder=1)
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Branching Distance (linkage)")
axes[0].set_title("Branching Distance")

ax2 = axes[0].twinx()
ax2.plot(n_clusters[:nshow], all_ratios[:nshow], marker='o', color="tab:orange", zorder=0)

nplots = np.min([nshow, 80])
axes[1].plot(n_clusters[:nplots]+1, diff_dist[:nplots], marker='o', zorder=1)
ax22 = axes[1].twinx()
ax22.plot(n_clusters[:nplots]+1, all_ratios[:nplots], color="tab:orange", zorder=0)

ax33 = axes[1].twinx()
ax33.plot(n_clusters[:nplots]+1, all_self[:nplots]/(n_clusters[:nplots]+1), color="tab:green", zorder=0)

axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Difference (Distance)")
axes[1].set_title("Difference in Branching Distance")

local_max_idx = argrelextrema(diff_dist, np.greater)[0]
ordered_maximas = local_max_idx[np.flip(np.argsort(diff_dist[local_max_idx]))]

print("Top 10 largest jumps in distance:")
for i, top10 in enumerate(ordered_maximas[:10]):
    print(f" -{i+1:02d}: K = {top10+2},\tdist: {diff_dist[top10]:.4f}, \tcut: {distances[top10]:.4f}")

    axes[1].scatter(n_clusters[top10]+1, diff_dist[top10], marker='o', color="tab:red", s=100)
    axes[1].text(n_clusters[top10]+1, diff_dist[top10]*1.01, f"K={top10+2}", fontsize=12,
                 ha='left', va='bottom')

### Figure

In [ ]:
fig, all_axes = plt.subplots(ncols=2, figsize=(25, 16), gridspec_kw={'wspace': 0, "width_ratios":[1, 1]})

all_axes[0].axis("off")
gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=all_axes[0].get_subplotspec(), hspace=0.1, height_ratios=[1, 2])
axes_left = [fig.add_subplot(gs[i]) for i in range(2)]

axes_left[0].axis("off")
gs = GridSpecFromSubplotSpec(1, 2, subplot_spec=axes_left[0].get_subplotspec(), wspace=0.1)
axes_mat = [fig.add_subplot(gs[i]) for i in range(2)]

axes_mat[0].spines[:].set_visible(False)
axes_mat[0].set_xticks([])
axes_mat[0].set_yticks([])
axes_mat[0].set_title("Directed Connectome", fontsize=18)
axes_mat[0].set_ylabel("Nodes", fontsize=16)
axes_mat[0].imshow(graph[order_by_lobe][:, order_by_lobe], cmap=cmaps["div_rb"], interpolation="none")
_, _, cbar = plot.add_cbar(fig, axes_mat[0])
cbar.set_ticks([0, 0.5, 1], labels=["0", "Bidirectional", "1"], rotation=90, ha="left", va="center")
cbar.ax.tick_params(labelsize=16)

axes_mat[1].spines[:].set_visible(False)
axes_mat[1].set_xticks([])
axes_mat[1].set_yticks([])
axes_mat[1].set_title("Bicommunity Consensus", fontsize=18)
axes_mat[1].set_ylabel("Edges", fontsize=16)
if not fast:
    axes_mat[1].imshow(1 - hierarchical_matrix, cmap=cmaps["div_rb"], interpolation="none")
    _, _, cbar = plot.add_cbar(fig, axes_mat[1], label="co-cluster probability")
cbar.set_ticks([0, 1], labels=["0", "1"], rotation=90, ha="left", va="center")
cbar.set_label("Co-cluster Probability", fontsize=16, va="bottom")
cbar.ax.tick_params(labelsize=16)

axes_left[1].axis("off")
axes_left[1].set_title("Hierarchical Clustering", fontsize=18)
gs = GridSpecFromSubplotSpec(1, 2, subplot_spec=axes_left[1].get_subplotspec(), wspace=0, width_ratios=[0.3, 1.5])
axes = [fig.add_subplot(gs[i]) for i in range(2)]

cut_dist = distances[:-1] - diff_dist/2
axes[0].plot(diff_dist, cut_dist, marker='o', markersize=4, lw=2, color="k")

selected_list = [-1, -2, -4]
selected_list = [-1, -9, -4]
selected_list = [0, 1, 2]

best10_list = ordered_maximas[selected_list]
if gamma == 3:
    x_dendro = [4.4e4, 4.5e4, 4.22e4]
else:
    x_dendro = [6.95e4, 6.6e4, 6.83e4]
for i, top10 in enumerate(best10_list):

    axes[0].scatter(diff_dist[top10], cut_dist[top10], marker='o', color="tab:red", s=50, zorder=4)
    axes[0].text(1.2*diff_dist.max(), cut_dist[top10]+7e-3, f"$K={top10+2}$", fontsize=14,
                 ha='left', va='bottom', zorder=5)
    
    axes[0].plot([diff_dist[top10], -0.1], [cut_dist[top10]]*2, color="r", lw=3, alpha=0.8, zorder=0)
    axes[1].axhline(cut_dist[top10], color="r", lw=3, alpha=0.8)

    axes[1].scatter(x_dendro[i], cut_dist[top10], s=200, color="none", edgecolors="r", lw=2)

cut_thresh = 0.77
cut_thresh = cut_dist[best10_list[0]]

axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].text(0.6 * diff_dist.max(), 0.42, "Hierarchical Distance",
             fontsize=14, rotation=90, va="bottom", ha="right")

axes[0].set_xlim(1.5*diff_dist.max(), -diff_dist.max()*0.1)
axes[0].spines[["top", "left", "right"]].set_visible(False)
axes[0].spines[:].set_linewidth(2)

axes[0].axis("off")
axes[1].axis("off")

e_labels = fcluster(Z, t=cut_thresh, criterion='distance')
k = e_labels.max()
ls_list = [{"lw": 1}]*len(Z)
cluster_cmap = cmaps["extended_ncar"].resampled(k+1)
d_dict, axes[1] = plot.styled_dendrogram(Z, color_threshold=0.5, cut_height=cut_thresh, lw=2, cmap=cluster_cmap, ax=axes[1], p=0.4)

axes[1].set_xlim(-500, np.max(d_dict["icoord"]) + 100)
axes[1].set_ylim(0.41, 1.0)
axes[0].set_ylim(0.41, 1.0)

selected_cuts = np.flip(sorted(cut_dist[best10_list]))

all_axes[1].axis("off")
gs = GridSpecFromSubplotSpec(len(selected_cuts), 1, subplot_spec=all_axes[1].get_subplotspec(), hspace=0.1)
axes_bicom = [fig.add_subplot(gs[i]) for i in range(len(selected_cuts))]

overlay_slines = False
opacity = 1
if not overlay_slines:
    opacity = 0.1

all_e_labels = []
all_e_mats = []
all_send_rec = []
all_conj = []
all_ks = []
all_dist = []
for i, axes_bc in enumerate(axes_bicom):
    axes_bc.axis("off")
    gs = GridSpecFromSubplotSpec(1, 3, subplot_spec=axes_bc.get_subplotspec(),
                                 wspace=0, width_ratios=[0.4, 1, 1])
    ax_bc = [fig.add_subplot(gs[i]) for i in range(3)]

    e_labels = fcluster(Z, t=selected_cuts[i], criterion='distance')
    print(e_labels.min(), e_labels.max())
    e_mat = np.zeros_like(graph, dtype=int)
    e_mat[graph != 0] = e_labels
    print(e_mat.min(), e_mat.max())

    send, rec = dgsp.get_node_clusters(e_labels, e_mat, scale=False)
    bimod_idx = dgsp.bimod_index_nodes(graph, send, rec, scale=True)
    sorting_array = np.flip(np.argsort(bimod_idx))

    all_e_labels.append(e_labels)
    all_e_mats.append(e_mat)
    all_send_rec.append((send, rec))

    ax_bc[0].axis("off")
    cluster_cmap = cmaps["cluster_palette_soft"]
    cluster_cmap = cmaps["extended_ncar"].resampled(e_labels.max()+1)
    ax_bc[1].imshow(e_mat[order_by_lobe][:, order_by_lobe], cmap=cluster_cmap, vmin=0, vmax=len(np.unique(e_labels))+1, interpolation="none")
    plot.plot_lobe_lines(ax_bc[1], lobe_sizes, lobe_labels, no_insula=True, x_hemi=True, fontsize=14)
    if i > 0:
        ax_bc[1].set_xticks([])

    sort_lab = np.argsort(e_labels)
    c_mat = 1 - hierarchical_matrix

    row, col, mat = dgsp.get_conjugates_matching(send, rec, unique=False, return_matrix=True)

    bic_id = [11, 17, 31][i]
    
    if not fast:
        plot.plot_bicom_tracts(
            bic_id,
            e_mat,
            labels,
            scale,
            cmap=cmaps["div_rb"],
            view="right",
            axes=ax_bc[2],
            n_centroids=5,
            linewidth=0.3,
            slines_alpha=0.99,
            brain_opacity=opacity,
            overlay_slines=overlay_slines,
            upsample=upsample,
            fig=fig,)
        
        if i == 2:
            plot.draw_cbar(fig, ax_pos=[0.76, 0.135, 0.12, 0.01],
                           cmap=cmaps["div_rb"], ticks=[0, 10],
                           labels=["Receive", "Send"])

# Plotting letters
axes_mat[0].text(0, 1.02, "A.", fontsize=18, transform=axes_mat[0].transAxes, weight="bold")
axes_mat[1].text(0, 1.02, "B.", fontsize=18, transform=axes_mat[1].transAxes, weight="bold")

axes[0].text(0.1, 1.02, "C.", fontsize=18, transform=axes[0].transAxes, weight="bold")

for ax, letter in zip(axes_bicom, ["D.", "E.", "F."]):
    ax.text(0.05, 1.02, letter, fontsize=18, transform=ax.transAxes, weight="bold")

for ax, letter in zip(axes_bicom, ["G.", "H.", "I."]):
    ax.text(0.6, 1.02, letter, fontsize=18, transform=ax.transAxes, weight="bold")

if not fast:
    upsample_suffix = f"_upsample{upsample}" if upsample is not None else ""
    trans_suffix = f"_trans{opacity}" if overlay_slines is not None else ""
    fig.savefig(op.join(path_to_figures, f"01-dendrogram-RIGHT{upsample_suffix}.png"),
                dpi=600, bbox_inches='tight')
    fig.savefig(op.join(path_to_figures, f"01-dendrogram-RIGHT{upsample_suffix}.pdf"),
                dpi=600, bbox_inches='tight', format="pdf")

## Adding Directionality

### Preparation Code

In [ ]:
bicom_id = 2

e_labels = all_e_labels[bicom_id]
edge_clusters_mat = all_e_mats[bicom_id]
e_clust = edge_clusters_mat[graph > 0]

print(edge_clusters_mat.max())

n_shuffle = 10000
perm_prop = 0.5

bicoms_masks = np.array([e_clust == (i+1) for i in range(e_clust.max())])
graph_asym = np.divide(graph, (graph + graph.T), where=(graph + graph.T) > 0, out=np.zeros_like(graph))
true_ratio = dgsp.get_asym_ratio(graph_asym[graph > 0], bicom_masks=bicoms_masks)
all_ratios_sym = np.zeros((n_shuffle, len(true_ratio)))

save_perm = True
# path_to_saveperm = f"./results/BrainKmeans/scale{scale}/permutations"
path_to_saveperm = op.join(path_to_derivatives, "permutations", f"scale{scale}")

fname = f"Dir-permutations_scale{scale}_gamma{gamma}-{n_shuffle}Perm-K{edge_clusters_mat.max()}"
fname += ".pkl"
print(f"Savename is {fname}")

if op.isfile(op.join(path_to_saveperm, fname)):
    print("Loading existing permutations...")
    perm_data = dload.load(op.join(path_to_saveperm, fname))
    all_shuffled = perm_data["all_shuffled"]
    all_ratios_sym = perm_data["all_ratios_sym"]
else:
    print("Computing new permutations...")
    all_shuffled = dgsp.shuffle_edges_sym(graph, perm_prop=perm_prop, n_shuffle=n_shuffle)

    for s_i, shuffled in enumerate(all_shuffled):
        shuffled_asym = np.divide(shuffled, (shuffled + shuffled.T),
                                    where=(shuffled + shuffled.T) > 0,
                                    out=np.zeros_like(shuffled))
        shuffled_ratio = dgsp.get_asym_ratio(shuffled_asym[graph > 0], bicom_masks=bicoms_masks)
        all_ratios_sym[s_i] = shuffled_ratio

    all_ratios_sym = np.nan_to_num(all_ratios_sym, nan=0.5)

    if save_perm:
        dload.save(op.join(path_to_saveperm, fname),
                {"all_shuffled": all_shuffled,
                    "all_ratios_sym": all_ratios_sym})

In [ ]:
bw_adjust = 0.1

perm_dist = all_ratios_sym.flatten()

print(f"Max p-val: {1 / (1 + n_shuffle)}")

all_ps = np.ones_like(true_ratio)
for i, t in enumerate(true_ratio):
    biggersum = np.sum(np.abs(all_ratios_sym[:, i] - all_ratios_sym[:, i].mean()) >= abs(t - all_ratios_sym[:, i].mean()))
    p_two = (1 + biggersum) / (1 + n_shuffle)
    all_ps[i] = p_two

sig_df = pd.DataFrame({
    "Bicomm_ID": np.arange(1, len(true_ratio)+1),
    "directionality": true_ratio,
    "abs_dir": np.abs(true_ratio - 0.5),
    "p_value": all_ps,
    "p_corr": all_ps * len(all_ps)
    })

sig_df["is_sig"] = sig_df["p_corr"] < 0.05
sig_df.sort_values(["is_sig", "abs_dir"], ascending=[False, False], inplace=True)
sig_df.head()

sort_by_p = sig_df["Bicomm_ID"].values - 1

sig_df

### Figure

In [ ]:
sort_by_p = sig_df["Bicomm_ID"].values - 1
rand_x = np.random.uniform(-0.3, 0.3, n_shuffle)
cluster_cmap = cmaps["extended_ncar"].resampled(edge_clusters_mat.max()+1)

if edge_clusters_mat.max() < 15:
    fig, all_axes = plt.subplots(nrows=2, figsize=(20, 10), gridspec_kw={"height_ratios": [1, 1.4], "hspace": 0.1})
else:
    fig, all_axes = plt.subplots(nrows=2, figsize=(20, 14), gridspec_kw={"height_ratios": [1, 1.8], "hspace": 0.1})

axpoly = fig.add_axes([0.1, 0.1, 0.8, 0.8], facecolor='none')

all_axes[0].axis("off")
gs = GridSpecFromSubplotSpec(1, 2, subplot_spec=all_axes[0].get_subplotspec(),
                             wspace=0.1, width_ratios=[1, 1.1])
axes = [fig.add_subplot(gs[0], facecolor="none"), fig.add_subplot(gs[1], facecolor="none")]

maxval = np.max([all_ratios_sym.max(), true_ratio.max()*1.02])

sig_mat = np.zeros_like(edge_clusters_mat)
non_sig_mat = np.zeros_like(edge_clusters_mat)

for i, sort_i in enumerate(sort_by_p):
    
    axes[1].scatter(rand_x + i, all_ratios_sym[:, sort_i], s=10,
                    color="gray", alpha=0.1, edgecolors="none")
    
    col = cluster_cmap(sort_i + 1)
    axes[1].bar(i, true_ratio[sort_i]-0.5, bottom=0.5, zorder=1,
             lw=2, edgecolor="none", color=col, alpha=0.5)
    axes[1].bar(i, true_ratio[sort_i]-0.5, bottom=0.5, zorder=1,
             lw=2, edgecolor=col, color="none")
    
    if sig_df.loc[sig_df["Bicomm_ID"] == (sort_i + 1), "is_sig"].values:
        axes[1].scatter(i, maxval, marker="$*$", color="k")

        sig_mat[edge_clusters_mat == (sort_i + 1)] = sort_i + 1
    else:
        non_sig_mat[edge_clusters_mat == (sort_i + 1)] = sort_i + 1

axes[1].axhline(0.5, lw=2, color="k")
axes[1].set_ylabel("Bicommunity Asymmetry", fontsize=16)
axes[1].spines[["top", "right", "bottom"]].set_visible(False)
axes[1].spines[:].set_linewidth(2)
axes[1].tick_params(which='both', bottom=False, labelbottom=False, width=2, labelsize=14)

if gamma == 3:
    axes[1].set_ylim(0.281, 0.699)
else:
    if edge_clusters_mat.max() < 15:
        maxval = np.max(np.abs(true_ratio - 0.5)) * 2
    else:
        maxval = np.max(np.abs(true_ratio - 0.5)) * 1.2
    axes[1].set_ylim(0.5 - maxval, 0.5 + maxval)

axes[0].axis("off")
gs = GridSpecFromSubplotSpec(3, 2, subplot_spec=axes[0].get_subplotspec(),
                             wspace=0, width_ratios=[1, 1], 
                             hspace=0, height_ratios=[0.05, 1, 0.2])
ax_matrices = [fig.add_subplot(gs[1, i]) for i in range(2)]

ax_matrices[0].set_title("Significantly Directed", fontsize=16, y=-0.12)
ax_matrices[1].set_title("Bidirectional", fontsize=16, y=-0.12)

ax_matrices[0].imshow(sig_mat[order_by_lobe][:, order_by_lobe], cmap=cluster_cmap,
            vmin=0, vmax=edge_clusters_mat.max(), interpolation="none")
plot.plot_lobe_lines(ax_matrices[0], lobe_sizes, lobe_labels, no_insula=True, x_hemi=True) #, y_only=True

ax_matrices[1].imshow(non_sig_mat[order_by_lobe][:, order_by_lobe], cmap=cluster_cmap,
               vmin=0, vmax=edge_clusters_mat.max(), interpolation="none")
plot.plot_lobe_lines(ax_matrices[1], lobe_sizes, lobe_labels, no_insula=True, x_only=True, x_hemi=True)
ax_matrices[1].set_yticks([])

all_axes[1].axis("off")

if edge_clusters_mat.max() < 15:
    n_bicom_to_show = 4
else:
    n_bicom_to_show = 8

if n_bicom_to_show > 4:
    gs = GridSpecFromSubplotSpec(4, n_bicom_to_show, subplot_spec=all_axes[1].get_subplotspec(), hspace=0, wspace=0)
    axes_tract = [fig.add_subplot(gs[0, i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract2 = [fig.add_subplot(gs[1, i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract += [fig.add_subplot(gs[2, i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract2 += [fig.add_subplot(gs[3, i], facecolor="none") for i in range(n_bicom_to_show//2)]
    
    axes_tract += [fig.add_subplot(gs[0, n_bicom_to_show//2 + i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract2 += [fig.add_subplot(gs[1, n_bicom_to_show//2 + i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract += [fig.add_subplot(gs[2, n_bicom_to_show//2 + i], facecolor="none") for i in range(n_bicom_to_show//2)]
    axes_tract2 += [fig.add_subplot(gs[3, n_bicom_to_show//2 + i], facecolor="none") for i in range(n_bicom_to_show//2)]
else:
    gs = GridSpecFromSubplotSpec(2, n_bicom_to_show*2, subplot_spec=all_axes[1].get_subplotspec(), hspace=0, wspace=0)
    axes_tract = [fig.add_subplot(gs[0, i], facecolor="none") for i in range(n_bicom_to_show*2)]
    axes_tract2 = [fig.add_subplot(gs[1, i], facecolor="none") for i in range(n_bicom_to_show*2)]

# Tract plot parameters
view = "custom2"
n_centroids = 5
linewidth = 0.3
slines_alpha = 0.99
brain_opacity = 0.1
overlay_slines = False

n_axes = len(axes_tract)
for i, ax in enumerate(axes_tract[:n_bicom_to_show]):
    bic_id = sig_df['Bicomm_ID'].iloc[i]
    p_plot = sig_df['p_corr'].iloc[i]
    asym = sig_df['directionality'].iloc[i]
    mat_cmap = LinearSegmentedColormap.from_list("bimask_cmap", ["k", "w", cluster_cmap(bic_id)], N=3)
    bimask = 2*(edge_clusters_mat == sig_df["Bicomm_ID"].iloc[i]).astype(int)
    bimask += (edge_clusters_mat > 0)

    e_clust_mask = np.zeros_like(edge_clusters_mat)
    if asym > 0.5:
        e_clust_mask[edge_clusters_mat == bic_id] = bic_id
    else:
        e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id

    if not fast:
        plot.plot_bicom_tracts(
            bic_id,
            e_clust_mask,
            labels,
            scale,
            cmap=cmaps["div_rb"],
            view="tra",
            n_centroids=n_centroids,
            linewidth=linewidth,
            slines_alpha=slines_alpha,
            brain_opacity=brain_opacity,
            overlay_slines=overlay_slines,
            axes=ax,
            upsample=upsample,
            fig=fig)
        
        ax.scatter([150], [180], marker="s", s=200, color=cluster_cmap(bic_id), edgecolor="k", lw=2)

        plot.plot_bicom_tracts(
            bic_id,
            e_clust_mask,
            labels,
            scale,
            cmap=cmaps["div_rb"],
            view="right",
            n_centroids=n_centroids,
            linewidth=linewidth,
            slines_alpha=slines_alpha,
            brain_opacity=brain_opacity,
            overlay_slines=overlay_slines,
            axes=axes_tract2[i],
            upsample=upsample,
            fig=fig)

for i, ax in enumerate(axes_tract[n_bicom_to_show:]):
    bic_id = sig_df['Bicomm_ID'].iloc[-n_bicom_to_show + i]
    p_plot = sig_df['p_corr'].iloc[-n_bicom_to_show + i]
    asym = sig_df['directionality'].iloc[-n_bicom_to_show + i]
    mat_cmap = LinearSegmentedColormap.from_list("bimask_cmap", ["k", "w", cluster_cmap(bic_id)], N=3)

    e_clust_mask = np.zeros_like(edge_clusters_mat)
    e_clust_mask[edge_clusters_mat == bic_id] = bic_id
    e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id

    if not fast:
        plot.plot_bicom_tracts(
            bic_id,
            e_clust_mask,
            labels,
            scale,
            cmap=cmaps["div_rb"],
            view="tra",
            n_centroids=n_centroids,
            linewidth=linewidth,
            slines_alpha=1,
            brain_opacity=brain_opacity,
            overlay_slines=overlay_slines,
            axes=ax,
            upsample=upsample,
            bidir_col=(0.6, 0.6, 0.6, 1),
            fig=fig)
        
        ax.scatter([150], [180], s=200, color=cluster_cmap(bic_id), edgecolor="none")

        plot.plot_bicom_tracts(
            bic_id,
            e_clust_mask,
            labels,
            scale,
            cmap=cmaps["div_rb"],
            view="right",
            n_centroids=n_centroids,
            linewidth=linewidth,
            slines_alpha=1,
            brain_opacity=brain_opacity,
            overlay_slines=overlay_slines,
            axes=axes_tract2[-n_bicom_to_show + i],
            upsample=upsample,
            bidir_col=(0.6, 0.6, 0.6, 1),
            fig=fig)

signif_start = 0.539
bidir_start = 0.978

if edge_clusters_mat.max() < 15:
    bicomspace = 0.027 * (1 + n_bicom_to_show)
    axpoly.fill([bidir_start, bidir_start - bicomspace, bidir_start - bicomspace, bidir_start],
                [0.737, 0.737, 0.74, 0.74], color="k", edgecolor="none")
    axpoly.text(signif_start + bicomspace/2, 0.65, "C", fontweight="bold", fontsize=14, ha="center")
    axpoly.text(bidir_start - bicomspace/2, 0.71, "D", fontweight="bold", fontsize=14, ha="center")
else:
    bicomspace = 0.011 * (1 + n_bicom_to_show)
    axpoly.fill([bidir_start, bidir_start - bicomspace, bidir_start - bicomspace, bidir_start],
                [0.767, 0.767, 0.77, 0.77], color="k", edgecolor="none")
    axpoly.text(signif_start + bicomspace/2, 0.66, "C", fontweight="bold", fontsize=14, ha="center")
    axpoly.text(bidir_start - bicomspace/2, 0.75, "D", fontweight="bold", fontsize=14, ha="center")

axpoly.fill([signif_start, signif_start + bicomspace,  signif_start + bicomspace, signif_start],
            [0.677, 0.677, 0.68, 0.68], color="k", edgecolor="none")

axpoly.set_xlim(0, 1)
axpoly.set_ylim(0, 1)
axpoly.axis("off")

axpoly.text(0.035, 0.97, "A.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)
axpoly.text(0.47, 0.97, "B.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)

if edge_clusters_mat.max() < 15:
    cbar = plot.draw_cbar(
    fig, cmap=cmaps["div_rb"], ax_pos=[0.115, 0.17, 0.01, 0.3],
    ticks=[0, 10], labels=["Receiving", "Sending"], fontsize=14,
    orientation="vertical")
    cbar.ax.tick_params(right=False, labelright=False, left=True, labelleft=True, rotation=90)
    cbar.ax.set_yticks([0, 10], labels=["Receiving", "Sending"], ha="right", va="center")
    
    axpoly.text(0.25, 0.54, "Significantly Directed", fontsize=20, ha='center', va='bottom', color='k')
    axpoly.text(0.75, 0.54, "Bidirectional", fontsize=20, ha='center', va='bottom', color='k')
    axpoly.text(0.035, 0.542, "C.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)
    axpoly.text(0.52, 0.542, "D.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)
else:
    cbar = plot.draw_cbar(
    fig, cmap=cmaps["div_rb"], ax_pos=[0.115, 0.17, 0.01, 0.38],
    ticks=[0, 10], labels=["Receiving", "Sending"], fontsize=14,
    orientation="vertical")
    cbar.ax.tick_params(right=False, labelright=False, left=True, labelleft=True, rotation=90)
    cbar.ax.set_yticks([0, 10], labels=["Receiving", "Sending"], ha="right", va="center")

    axpoly.text(0.25, 0.611, "Significantly Directed", fontsize=20, ha='center', va='bottom', color='k')
    axpoly.text(0.75, 0.611, "Bidirectional", fontsize=20, ha='center', va='bottom', color='k')
    axpoly.text(0.035, 0.615, "C.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)
    axpoly.text(0.52, 0.615, "D.", fontsize=18, fontweight='bold', ha='left', va='bottom', transform=axpoly.transAxes)

if not fast:
    upsample_suffix = f"_upsample{upsample}" if upsample is not None else ""
    fig.savefig(op.join(path_to_figures, f"02-Directionality{upsample_suffix}_K{edge_clusters_mat.max()}-show{n_bicom_to_show}.png"),
                dpi=600, bbox_inches='tight')
    fig.savefig(op.join(path_to_figures, f"02-Directionality{upsample_suffix}_K{edge_clusters_mat.max()}-show{n_bicom_to_show}.pdf"),
                dpi=600, bbox_inches='tight', format="pdf")

## Network Characterization

### Preparation Code

In [ ]:
n_per_block = 12

my_graph = np.random.normal(loc=1, scale=1, size=(n_per_block*2, n_per_block*2))
rand_thresh = 1
my_graph = my_graph * (my_graph > rand_thresh)

my_sends = np.array(
    [
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.ones(n_per_block+n_per_block//2), np.zeros(n_per_block-n_per_block//2)]),
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.zeros(n_per_block-n_per_block//4), np.ones(n_per_block-n_per_block//2), np.zeros(n_per_block-n_per_block//4)]),
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
    ]
)

my_recs = np.array(
    [
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.ones(n_per_block+n_per_block//2), np.zeros(n_per_block-n_per_block//2)]),
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.zeros(n_per_block-n_per_block//4), np.ones(n_per_block-n_per_block//2), np.zeros(n_per_block-n_per_block//4)]),
        np.concatenate([np.ones(n_per_block), np.zeros(n_per_block)]),
        np.concatenate([np.zeros(n_per_block), np.ones(n_per_block)]),
    ]
)


ss_sr_toy = np.zeros(len(my_sends))
rr_sr_toy = np.zeros(len(my_sends))

all_sample_graphs = []
for i in range(len(my_sends)):
    bic_id = i
    mymat = edge_clusters_mat.copy()
    mymat[mymat != (bic_id + 1)] = 0

    ss = np.outer(my_sends[bic_id], my_sends[bic_id])
    rr = np.outer(my_recs[bic_id], my_recs[bic_id])

    sr = np.outer(my_sends[bic_id], my_recs[bic_id]) * my_graph
    all_sample_graphs.append(sr)

    ss_sr_toy[i] = np.sum(ss * sr)/np.sum(ss)
    rr_sr_toy[i] = np.sum(rr * sr)/np.sum(rr)

fig, axes = plt.subplots(nrows=2, figsize=(15, 8), gridspec_kw={"height_ratios": [1, 2], "hspace":0})

for ax in axes:
    ax.axis("off")

gs = GridSpecFromSubplotSpec(1, len(my_sends), subplot_spec=axes[0].get_subplotspec(), wspace=0.1)
ax_mat = [fig.add_subplot(gs[i]) for i in range(len(my_sends))]

gs = GridSpecFromSubplotSpec(1, 2, subplot_spec=axes[1].get_subplotspec(), wspace=0.1, width_ratios=[1.8, 1])
ax_plot = fig.add_subplot(gs[0])
ax_scat = fig.add_subplot(gs[1])

ax_plot.plot(np.arange(len(my_sends))+1, ss_sr_toy, label="Send-SR overlap", lw=2, color="tab:red", alpha=0.8)
ax_plot.plot(np.arange(len(my_sends))+1, rr_sr_toy, label="Rec-SR overlap", lw=2, color="tab:blue", alpha=0.8)
ax_plot.plot(np.arange(len(my_sends))+1, (ss_sr_toy+rr_sr_toy)/2, label="Mean", lw=4, color="gray", alpha=0.8, zorder=0)

ax_plot.legend(fontsize=14)

mean_assort = (ss_sr_toy + rr_sr_toy)/2
ax_scat.scatter(mean_assort, rr_sr_toy - ss_sr_toy, s=300, edgecolor="k", c=np.arange(len(ss_sr_toy))+1,
                cmap=cluster_cmap, vmin=0, vmax=len(ss_sr_toy)+1, lw=2)
maxval = np.max([ss_sr_toy.max(), rr_sr_toy.max()])

for i in range(len(my_sends)):
    ax_scat.text(mean_assort[i] + mean_assort.max()*0, rr_sr_toy[i] - ss_sr_toy[i], f"{i+1:d}", ha="center", va="center", fontsize=14, fontweight='bold')
for i, ax in enumerate(ax_mat):
    mymat = np.zeros_like(my_graph)
    mymat = (my_graph > rand_thresh) * np.outer(my_sends[i] > 0, my_recs[i] > 0).astype(float) * (i+1)

    sort_by_sendrec = np.flip(np.argsort(my_sends[i]))
    ax.imshow(mymat, cmap=cluster_cmap, vmin=0, vmax=len(my_sends)+1, interpolation="none")
    ax.set_title(f"#{i+1}", fontsize=16)
    ax.axis("off")


In [ ]:
send, rec = all_send_rec[bicom_id]
bimod_idx = dgsp.bimod_index_nodes(graph, send, rec)

send = send / send.sum(axis=1, keepdims=True)
rec = rec / rec.sum(axis=1, keepdims=True)

ss_sr = np.zeros(len(send))
rr_sr = np.zeros(len(send))

for i in range(len(send)):
    ss = np.outer(send[i], send[i])
    rr = np.outer(rec[i], rec[i])
    
    sr = graph.copy()
    sr[edge_clusters_mat != (i + 1)] = 0

    ss_sr[i] = np.sum(ss * sr)/np.sum(ss)
    rr_sr[i] = np.sum(rr * sr)/np.sum(rr)

assortativity = (ss_sr + rr_sr)/2
sendrec = (rr_sr - ss_sr)

sig_df["assortativity"] = assortativity[sort_by_p]
sig_df["sendrec"] = sendrec[sort_by_p]
sig_df["bimodality_index"] = bimod_idx[sort_by_p]

is_assort = sig_df["assortativity"] >= 1e-1

### Figure

In [ ]:
from matplotlib.patches import FancyArrowPatch

fig, main_ax = plt.subplots(figsize=(30, 16))
lobe_summary = False

all_a_pos = [
    [[-0.1, 0], [0.5, 0]],
    [[0.1, 0], [-0.5, 0]],
    [[0, -0.1], [0, 0.95]],
    [[0, 0.1], [0, -0.9]],
]
for a_pos in all_a_pos:
    arrow = FancyArrowPatch(
        a_pos[0],
        a_pos[1],
        mutation_scale=40,
        facecolor="k",
        edgecolor="none",
        arrowstyle="simple",
    )
    main_ax.add_patch(arrow)

clust_cmap_net = cluster_cmap.resampled(edge_clusters_mat.max()+1)

main_ax.text(0.32, 0.02, "Assortative", fontsize=22, ha='left', va='bottom', color='k')
main_ax.text(-0.32, 0.02, "Disassortative", fontsize=22, ha='right', va='bottom', color='k')
main_ax.text(0, 0.95, "Sending", fontsize=22, ha='center', va='bottom', color='k')
main_ax.text(0, -0.9, "Receiving", fontsize=22, ha='center', va='top', color='k')

main_ax.set_xlim(-1, 1)
main_ax.set_ylim(-1.1, 1.1)
main_ax.axis("off")

gs = GridSpecFromSubplotSpec(1, 3, subplot_spec=main_ax.get_subplotspec(), wspace=0.1, width_ratios=[2, 1.5, 2])
axes = [fig.add_subplot(gs[i]) for i in range(3)]

for ax in axes:
    ax.axis("off")
gs = GridSpecFromSubplotSpec(4, 2, subplot_spec=axes[1].get_subplotspec(), hspace=0.1, wspace=0.2)

gs_disassort = GridSpecFromSubplotSpec(5, 3, subplot_spec=axes[0].get_subplotspec(),
                                       hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0.2, 1, 1])
gs_assort = GridSpecFromSubplotSpec(5, 3, subplot_spec=axes[2].get_subplotspec(),
                                    hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0.2, 1, 1])
axes_bicom = [fig.add_subplot(gs_disassort[i], facecolor="none") for i in range(15) if i//3 != 2]
axes_bicom += [fig.add_subplot(gs_assort[i], facecolor="none") for i in range(15) if i//3 != 2]

send, rec = all_send_rec[bicom_id]
assortativity = np.diag(send @ rec.T)
bimod_idx = dgsp.bimod_index_nodes(graph, send, rec)

if gamma == 3:
    assort_bicom = [11, 13, 8, 21, 26, 1, 16, 24, 25, 5, 17, 7]
    disassort_bicom = [6, 12, 27, 10, 9, 0, 20, 22, 23, 15, 14, 4]
else:
    assort_bicom = [1, 18, 9, 1, 1, 1, 5, 15, 6, 13, 14, 14]
    disassort_bicom = [11, 12, 10, 8, 16, 0, 3, 16, 4, 17, 7]

    assort_bicom = [19, 13, 3, 14, 34, 27, 6, 33, 5, 12, 29, 25]
    disassort_bicom = [15, 8, 20, 17, 4, 28, 24, 10, 11, 16, 23, 1]

lobe_vec = lobe_df.lobe_id_reorder.values
one_hot_lobe = np.array([(lobe_vec == i).astype(float) for i in range(lobe_vec.max())])
one_hot_lobe /= one_hot_lobe.sum(axis=1, keepdims=True)

assort_mask = [False]*len(disassort_bicom[:12]) + [True]*len(assort_bicom[:12])
selected_bicoms = disassort_bicom[:12] + assort_bicom[:12]

# If needed to adjust for 0-indexing
selected_bicoms = [x + 1 for x in selected_bicoms]

i_assort = 0
i_disassort = 0
for i, bic_id in enumerate(selected_bicoms):
    if assort_mask[i]:
        ax = axes_bicom[i_assort+12]
        i_assort += 1
    else:
        ax = axes_bicom[i_disassort]
        i_disassort += 1

    e_clust_mask = np.zeros_like(edge_clusters_mat)
    e_clust_mask[edge_clusters_mat == bic_id] = bic_id

    if lobe_summary:
        lobe_repr = one_hot_lobe @ (e_clust_mask > 0).astype(float) @ one_hot_lobe.T

        indiv_cmap = LinearSegmentedColormap.from_list("indiv_cmap", ["k", clust_cmap_net(bic_id)], N=100)
        ax.imshow(lobe_repr, cmap=indiv_cmap, interpolation="none")
        scatter_pos = (10, 1)
    else:
        ax.imshow(e_clust_mask[order_by_lobe][:, order_by_lobe], cmap=clust_cmap_net, vmin=0,
                vmax=edge_clusters_mat.max(), interpolation="none")
        scatter_pos = (120, 10)
        
    if sig_df.loc[bic_id-1, "is_sig"]:
        ax.scatter(scatter_pos[0], scatter_pos[1], s=120, color="w", marker="$*$", zorder=2)

    if assortativity[bic_id-1] == 0:
        ax.scatter(scatter_pos[1], scatter_pos[1], s=80, color="w", marker="D", zorder=2)
    
for ax in axes_bicom:
    ax.axis("off")

ax_scatter = fig.add_subplot(gs[1:3, 0:2], facecolor="none")

scatter_size = 300 + 2*bimod_idx
sig_mask = sig_df.sort_values("Bicomm_ID").is_sig.values

assort_plot = assortativity.copy()
min_assort = 1e-2
np.random.seed(17)

mean_assort = (ss_sr + rr_sr)/2
assort_plot = mean_assort

ax_scatter.scatter(assort_plot[~sig_mask], (rr_sr - ss_sr)[~sig_mask], s=scatter_size[~sig_mask],
                   c=np.arange(len(sig_mask))[~sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                   vmin=0, vmax=len(sig_mask))
ax_scatter.scatter(assort_plot[sig_mask], (rr_sr - ss_sr)[sig_mask], s=scatter_size[sig_mask], marker="s",
                   c=np.arange(len(sig_mask))[sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                   vmin=0, vmax=len(sig_mask))

ax_scatter.scatter(assort_plot[sig_mask], (rr_sr - ss_sr)[sig_mask], s=scatter_size[sig_mask]/3, marker="$*$",
                   color="w", edgecolors="w")

ax_scatter.axhline(0, color="k", lw=2, zorder=0)
ax_scatter.axvline(0.1, color="k", lw=2, zorder=0)

limit_ratio = 1.2
ax_scatter.set_xlim(-mean_assort.max()*limit_ratio + 0.2, mean_assort.max()*limit_ratio)
ax_scatter.set_ylim(-np.abs(rr_sr - ss_sr).max()*limit_ratio, np.abs(rr_sr - ss_sr).max()*limit_ratio)

ax_in = ax_scatter.inset_axes([0.05, 0.7, 0.3, 0.3], xlim=(-1e-3, 1e-3), ylim=(-1e-3, 1e-3))

zero_mask = mean_assort < 1e-3
np.random.seed(7)
zero_x = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())
zero_y = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())

ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="D",
              c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
              vmin=0, vmax=len(sig_mask), alpha=(sig_mask[zero_mask] == False).astype(float))

ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="D",
              c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
              vmin=0, vmax=len(sig_mask), alpha=sig_mask[zero_mask].astype(float))

ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask]/3, marker="$*$",
              color="w", edgecolors="w", alpha=sig_mask[zero_mask].astype(float))

ax_in.set_xticks([])
ax_in.set_yticks([])
ax_in.set_title("Assortativity = 0", fontsize=16)
ax_in.spines[:].set_linewidth(2)
ax_scatter.indicate_inset_zoom(ax_in, edgecolor="black", lw=2, zorder=0)
ax_scatter.axis("off")

toy_pos = [
    [0.56, 0.46, 0.1, 0.08],
    [0.53, 0.2, 0.1, 0.08],
    [0.53, 0.71, 0.1, 0.08],
    [0.39, 0.71, 0.1, 0.08],
    [0.39, 0.2, 0.1, 0.08],
    [0.368, 0.46, 0.1, 0.08],
]
toy_titles = [
    "",
    "Receiving\nCore-Periphery",
    "Sending\nCore-Periphery",
    "Disassortative\nFocusing",
    "Disassortative\nBroadcasting",
    "",
]
for toy_i, (toy_mat, pos) in enumerate(zip(all_sample_graphs, toy_pos)):
    ax_toy = fig.add_subplot(pos, facecolor="none")
    ax_toy.imshow((toy_mat > 0).astype(int), cmap=cmaps["div_rb"],
                  interpolation="none", vmax=1.3)
    ax_toy.set_title(toy_titles[toy_i], fontsize=18)
    ax_toy.axis("off")

l1_pos = -0.25
main_ax.scatter([l1_pos-0.02], [-1.04], marker="D", s=300, color="gray", edgecolor="k", lw=1)
main_ax.text(l1_pos, -1.045, "Purely Disassortative", fontsize=16, ha="left", va="center")

l2_pos = 0.08
main_ax.scatter([l2_pos-0.02], [-1.04], marker="s", s=400, color="gray", edgecolor="k", lw=2)
main_ax.scatter([l2_pos-0.02], [-1.04], marker="$*$", s=100, color="w")
main_ax.text(l2_pos, -1.045, "Significantly Directed", fontsize=16, ha="left", va="center")

# lobe_sum_suffix = "_ByLobe" if lobe_summary else ""
# fig.savefig(op.join(path_to_figures, f"03-Showcase-MATRICES_K{edge_clusters_mat.max()}{lobe_sum_suffix}{fig_suffix}.png"),
#             dpi=600, bbox_inches='tight')
# fig.savefig(op.join(path_to_figures, f"03-Showcase-MATRICES_K{edge_clusters_mat.max()}{lobe_sum_suffix}{fig_suffix}.pdf"),
#             dpi=600, bbox_inches='tight', format="pdf")

In [ ]:
if edge_clusters_mat.max() < 20:
    fig, main_ax = plt.subplots(figsize=(25, 16))

    all_a_pos = [
        [[-0.1, 0], [0.5, 0]],
        [[0.1, 0], [-0.5, 0]],
        [[0, -0.1], [0, 0.95]],
        [[0, 0.1], [0, -0.9]],
    ]
    for a_pos in all_a_pos:
        arrow = FancyArrowPatch(
            a_pos[0],
            a_pos[1],
            mutation_scale=40,
            facecolor="k",
            edgecolor="none",
            arrowstyle="simple",
        )
        main_ax.add_patch(arrow)

    main_ax.text(0.32, 0.02, "Assortative", fontsize=22, ha='left', va='bottom', color='k')
    main_ax.text(-0.32, 0.02, "Disassortative", fontsize=22, ha='right', va='bottom', color='k')
    main_ax.text(0, 0.95, "Sending", fontsize=22, ha='center', va='bottom', color='k')
    main_ax.text(0, -0.9, "Receiving", fontsize=22, ha='center', va='top', color='k')

    main_ax.set_xlim(-1, 1)
    main_ax.set_ylim(-1.1, 1.1)
    main_ax.axis("off")

    gs = GridSpecFromSubplotSpec(1, 3, subplot_spec=main_ax.get_subplotspec(), wspace=0.1, width_ratios=[2, 1.8, 2])
    axes = [fig.add_subplot(gs[i]) for i in range(3)]

    for ax in axes:
        ax.axis("off")
    gs = GridSpecFromSubplotSpec(4, 2, subplot_spec=axes[1].get_subplotspec(), hspace=0.1, wspace=0.2)
    gs_disassort = GridSpecFromSubplotSpec(5, 2, subplot_spec=axes[0].get_subplotspec(),
                                           hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0.1, 1, 1])
    gs_assort = GridSpecFromSubplotSpec(5, 2, subplot_spec=axes[2].get_subplotspec(),
                                        hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0.1, 1, 1])
    axes_bicom = [fig.add_subplot(gs_disassort[i], facecolor="none") for i in range(10) if i//2 != 2]
    axes_bicom += [fig.add_subplot(gs_assort[i], facecolor="none") for i in range(10) if i//2 != 2]

    send, rec = all_send_rec[bicom_id]
    assortativity = np.diag(send @ rec.T)
    bimod_idx = dgsp.bimod_index_nodes(graph, send, rec)

    assort_bicom = [1, 18, 9, 1, 15, 6, 13, 14]
    disassort_bicom = [12, 10, 8, 0, 16, 4, 17, 7]
    
    manual_view = [
        "tra", "left", "tra", "left",
        "right", "left", "right", "tra",
        "left", "right", "tra", "left",
        "right", "left", "right", "right",
    ]

    lobe_vec = lobe_df.lobe_id_reorder.values
    one_hot_lobe = np.array([(lobe_vec == i).astype(float) for i in range(lobe_vec.max())])
    one_hot_lobe /= one_hot_lobe.sum(axis=1, keepdims=True)

    assort_mask = [False]*len(disassort_bicom[:8]) + [True]*len(assort_bicom[:8])
    selected_bicoms = disassort_bicom[:8] + assort_bicom[:8]

    # If needed to adjust for 0-indexing
    selected_bicoms = [x + 1 for x in selected_bicoms]
    # manual_view = ["tra"]* len(selected_bicoms)

    i_assort = 0
    i_disassort = 0
    for i, bic_id in enumerate(selected_bicoms):
        if assort_mask[i]:
            ax = axes_bicom[i_assort+8]
            i_assort += 1
        else:
            ax = axes_bicom[i_disassort]
            i_disassort += 1

        is_sig = sig_df.loc[bic_id-1, "is_sig"]
        assym = sig_df.loc[bic_id-1, "directionality"]
        e_clust_mask = np.zeros_like(edge_clusters_mat)

        col = "w"
        marker = "o"
        lw = 0

        fontweight=None
        if is_sig:
            fontweight="bold"
            marker = "s"
            lw = 2

            if assym < 0.5:
                e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id
            else:
                e_clust_mask[edge_clusters_mat == bic_id] = bic_id
        else: 
            e_clust_mask[edge_clusters_mat == bic_id] = bic_id
            e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id

        ax.scatter([200], [200], s=500, color=clust_cmap_net(bic_id),
                edgecolor="k", marker=marker, lw=lw, zorder=1)


        if not fast:
            plot.plot_bicom_tracts(
                bic_id,
                e_clust_mask,
                labels,
                scale,
                cmap=cmaps["div_rb"],
                view=manual_view[i],
                axes=ax,
                n_centroids=5,
                linewidth=0.3,
                slines_alpha=1 - 0.01 * is_sig,
                upsample=upsample,
                overlay_slines=False,
                brain_opacity=0.1,
                bidir_col=(0.6, 0.6, 0.6, 1),
                fig=fig)
        
    for ax in axes_bicom:
        ax.axis("off")

    ax_scatter = fig.add_subplot(gs[1:3, 0:2], facecolor="none")

    scatter_size = 300 + 2*bimod_idx
    sig_mask = sig_df.sort_values("Bicomm_ID").is_sig.values

    assort_plot = assortativity.copy()
    min_assort = 1e-2
    np.random.seed(17)

    mean_assort = (ss_sr + rr_sr)/2
    assort_plot = mean_assort

    ax_scatter.scatter(assort_plot[~sig_mask], (rr_sr - ss_sr)[~sig_mask], s=scatter_size[~sig_mask],
                    c=np.arange(len(sig_mask))[~sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=0,
                    vmin=0, vmax=len(sig_mask))
    ax_scatter.scatter(assort_plot[sig_mask], (rr_sr - ss_sr)[sig_mask], s=scatter_size[sig_mask], marker="s",
                    c=np.arange(len(sig_mask))[sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                    vmin=0, vmax=len(sig_mask))

    ax_scatter.axhline(0, color="k", lw=2, zorder=0)
    ax_scatter.axvline(0.1, color="k", lw=2, zorder=0)

    limit_ratio = 1.2
    ax_scatter.set_xlim(-mean_assort.max()*limit_ratio + 0.2, mean_assort.max()*limit_ratio)
    ax_scatter.set_ylim(-np.abs(rr_sr - ss_sr).max()*limit_ratio, np.abs(rr_sr - ss_sr).max()*limit_ratio)

    ax_in = ax_scatter.inset_axes([0.05, 0.7, 0.3, 0.3], xlim=(-1e-3, 1e-3), ylim=(-1e-3, 1e-3))

    zero_mask = mean_assort < 1e-3
    np.random.seed(7)
    zero_x = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())
    zero_y = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())

    ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="o", #marker="D",
                c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=0,
                vmin=0, vmax=len(sig_mask), alpha=(sig_mask[zero_mask] == False).astype(float))

    ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="s", #marker="D",
                c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                vmin=0, vmax=len(sig_mask), alpha=sig_mask[zero_mask].astype(float))

    ax_in.set_xticks([])
    ax_in.set_yticks([])
    ax_in.set_title("Assortativity = 0", fontsize=16)
    ax_in.spines[:].set_linewidth(2)
    ax_scatter.indicate_inset_zoom(ax_in, edgecolor="black", lw=2, zorder=0)
    ax_scatter.axis("off")

    toy_pos = [
        [0.56, 0.46, 0.1, 0.08],
        [0.53, 0.2, 0.1, 0.08],
        [0.53, 0.71, 0.1, 0.08],
        [0.39, 0.71, 0.1, 0.08],
        [0.39, 0.2, 0.1, 0.08],
        [0.368, 0.46, 0.1, 0.08],
    ]
    toy_titles = [
        "",
        "Receiving\nCore-Periphery",
        "Sending\nCore-Periphery",
        "Disassortative\nFocusing",
        "Disassortative\nBroadcasting",
        "",
    ]
    for toy_i, (toy_mat, pos) in enumerate(zip(all_sample_graphs, toy_pos)):
        ax_toy = fig.add_subplot(pos, facecolor="none")
        ax_toy.imshow((toy_mat > 0).astype(int), cmap=cmaps["div_rb"],
                    interpolation="none", vmax=1.3)
        ax_toy.set_title(toy_titles[toy_i], fontsize=18)
        ax_toy.set_xticks([])
        ax_toy.set_yticks([])
        ax_toy.axis("off")

    l1_pos = -0.21
    main_ax.scatter([l1_pos-0.02], [-1], marker="o", s=400, color="gray", edgecolor="k", lw=0)
    main_ax.text(l1_pos, -1.005, "Bidirectional", fontsize=16, ha="left", va="center")

    l2_pos = 0.02
    main_ax.scatter([l2_pos-0.02], [-1], marker="s", s=400, color="gray", edgecolor="k", lw=2)
    main_ax.text(l2_pos, -1.005, "Significantly Directed", fontsize=16, ha="left", va="center")

    cbar_pos = [[0.4, 0.11, 0.23, 0.01]]

    for pos in cbar_pos:
        plot.draw_cbar(fig, cmap=cmaps["div_rb"], ax_pos=pos, ticks=[0, 10],
                    labels=["Receiving", "Sending"], fontsize=16)

    # Drawing boxes
    box_lw = 2
    axe_boxes = fig.add_subplot([0.1, 0.1, 0.81, 0.81], facecolor="none")
    axe_boxes.axis("off")
    axe_boxes.fill([0.03, 0.338, 0.338, 0.03], [0.525, 0.525, 0.96, 0.96], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.03, 0.338, 0.338, 0.03], [0.04, 0.04, 0.467, 0.467], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.68, 0.99, 0.99, 0.68], [0.525, 0.525, 0.96, 0.96], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.68, 0.99, 0.99, 0.68], [0.04, 0.04, 0.467, 0.467], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.set_xlim(0, 1)
    axe_boxes.set_ylim(0, 1)

    main_ax.plot([-0.356, -0.24], [0.7, 0.74], color="k", lw=box_lw)
    main_ax.plot([0.357, 0.23], [0.7, 0.74], color="k", lw=box_lw)
    main_ax.plot([-0.356, -0.24], [-0.7, -0.74], color="k", lw=box_lw)
    main_ax.plot([0.357, 0.23], [-0.7, -0.74], color="k", lw=box_lw)

    main_ax.plot([0.357, 0.255], [0.24, 0.13], color="k", lw=box_lw)
    main_ax.plot([-0.356, -0.25], [0.24, 0.13], color="k", lw=box_lw)
    main_ax.plot([0.357, 0.255], [-0.21, -0.1], color="k", lw=box_lw)
    main_ax.plot([-0.356, -0.25], [-0.21, -0.1], color="k", lw=box_lw)

    if not fast:
        fig.savefig(op.join(path_to_figures, f"03-Showcase-BICOM_K{edge_clusters_mat.max()}{fig_suffix}.png"),
                    dpi=600, bbox_inches='tight')
        fig.savefig(op.join(path_to_figures, f"03-Showcase-BICOM_K{edge_clusters_mat.max()}{fig_suffix}.pdf"),
                    dpi=600, bbox_inches='tight', format="pdf")

In [ ]:
if edge_clusters_mat.max() > 20:
    fig, main_ax = plt.subplots(figsize=(30, 16))

    all_a_pos = [
        [[-0.1, 0], [0.5, 0]],
        [[0.1, 0], [-0.5, 0]],
        [[0, -0.1], [0, 0.95]],
        [[0, 0.1], [0, -0.9]],
    ]
    for a_pos in all_a_pos:
        arrow = FancyArrowPatch(
            a_pos[0],
            a_pos[1],
            mutation_scale=40,
            facecolor="k",
            edgecolor="none",
            arrowstyle="simple",
        )
        main_ax.add_patch(arrow)

    main_ax.text(0.32, 0.02, "Assortative", fontsize=22, ha='left', va='bottom', color='k')
    main_ax.text(-0.32, 0.02, "Disassortative", fontsize=22, ha='right', va='bottom', color='k')
    main_ax.text(0, 0.95, "Sending", fontsize=22, ha='center', va='bottom', color='k')
    main_ax.text(0, -0.9, "Receiving", fontsize=22, ha='center', va='top', color='k')

    main_ax.set_xlim(-1, 1)
    main_ax.set_ylim(-1.1, 1.1)
    main_ax.axis("off")

    gs = GridSpecFromSubplotSpec(1, 3, subplot_spec=main_ax.get_subplotspec(), wspace=0.1, width_ratios=[2, 1.5, 2])
    axes = [fig.add_subplot(gs[i]) for i in range(3)]

    for ax in axes:
        ax.axis("off")
    gs = GridSpecFromSubplotSpec(4, 2, subplot_spec=axes[1].get_subplotspec(), hspace=0.1, wspace=0.2)
    gs_disassort = GridSpecFromSubplotSpec(5, 3, subplot_spec=axes[0].get_subplotspec(),
                                        hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0, 1, 1])
    gs_assort = GridSpecFromSubplotSpec(5, 3, subplot_spec=axes[2].get_subplotspec(),
                                        hspace=0.02, wspace=0.02, height_ratios=[1, 1, 0, 1, 1])
    axes_bicom = [fig.add_subplot(gs_disassort[i], facecolor="none") for i in range(15) if i//3 != 2]
    axes_bicom += [fig.add_subplot(gs_assort[i], facecolor="none") for i in range(15) if i//3 != 2]

    send, rec = all_send_rec[bicom_id]
    assortativity = np.diag(send @ rec.T)
    bimod_idx = dgsp.bimod_index_nodes(graph, send, rec)

    if gamma == 3:
        assort_bicom = [11, 13, 8, 21, 26, 1, 16, 24, 25, 5, 17, 7]
        disassort_bicom = [6, 12, 27, 10, 9, 0, 20, 22, 23, 15, 14, 4]

        manual_view = [
            "right", "left", "tra", "tra", "right", "right",
            "left", "tra", "left", "tra", "tra", "right",
            "right", "left", "right", "left", "tra", "right",
            "left", "left", "left", "right", "left", "right"
        ]
    else:
        assort_bicom = [19, 13, 3, 14, 34, 27, 6, 33, 5, 12, 29, 26]
        disassort_bicom = [15, 8, 20, 17, 4, 28, 24, 10, 11, 16, 23, 1]
        manual_view = [
            "tra", "right", "left", "right", "left", "right",
            "tra", "left", "left", "left", "tra", "left",
            "tra", "left", "right", "left", "right", "right",
            "left", "right", "left", "left", "right", "right"
        ]

    lobe_vec = lobe_df.lobe_id_reorder.values
    one_hot_lobe = np.array([(lobe_vec == i).astype(float) for i in range(lobe_vec.max())])
    one_hot_lobe /= one_hot_lobe.sum(axis=1, keepdims=True)

    assort_mask = [False]*len(disassort_bicom[:12]) + [True]*len(assort_bicom[:12])
    selected_bicoms = disassort_bicom[:12] + assort_bicom[:12]

    # If needed to adjust for 0-indexing
    selected_bicoms = [x + 1 for x in selected_bicoms]

    i_assort = 0
    i_disassort = 0
    for i, bic_id in enumerate(selected_bicoms):
        if assort_mask[i]:
            ax = axes_bicom[i_assort+12]
            i_assort += 1
        else:
            ax = axes_bicom[i_disassort]
            i_disassort += 1
        
        is_sig = sig_df.loc[bic_id-1, "is_sig"]
        assym = sig_df.loc[bic_id-1, "directionality"]
        e_clust_mask = np.zeros_like(edge_clusters_mat)

        col = "w"
        marker = "o"
        lw = 0

        fontweight=None
        if is_sig:
            fontweight="bold"
            marker = "s"
            lw = 2

            if assym < 0.5:
                e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id
            else:
                e_clust_mask[edge_clusters_mat == bic_id] = bic_id
        else: 
            e_clust_mask[edge_clusters_mat == bic_id] = bic_id
            e_clust_mask[edge_clusters_mat.T == bic_id] = bic_id

        ax.scatter([200], [200], s=500, color=clust_cmap_net(bic_id),
                edgecolor="k", marker=marker, lw=lw, zorder=1)

        if not fast:
            plot.plot_bicom_tracts(
                bic_id,
                e_clust_mask,
                labels,
                scale,
                cmap=cmaps["div_rb"],
                view=manual_view[i],
                axes=ax,
                n_centroids=5,
                linewidth=0.3,
                slines_alpha=1 - 0.01 * is_sig,
                upsample=upsample,
                overlay_slines=False,
                brain_opacity=0.1,
                bidir_col=(0.6, 0.6, 0.6, 1),
                fig=fig)
        
    for ax in axes_bicom:
        ax.axis("off")

    ax_scatter = fig.add_subplot(gs[1:3, 0:2], facecolor="none")

    scatter_size = 300 + 2*bimod_idx
    sig_mask = sig_df.sort_values("Bicomm_ID").is_sig.values

    assort_plot = assortativity.copy()
    min_assort = 1e-2
    np.random.seed(17)

    mean_assort = (ss_sr + rr_sr)/2
    assort_plot = mean_assort

    ax_scatter.scatter(assort_plot[~sig_mask], (rr_sr - ss_sr)[~sig_mask], s=scatter_size[~sig_mask],
                    c=np.arange(len(sig_mask))[~sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=0,
                    vmin=0, vmax=len(sig_mask))
    ax_scatter.scatter(assort_plot[sig_mask], (rr_sr - ss_sr)[sig_mask], s=scatter_size[sig_mask], marker="s",
                    c=np.arange(len(sig_mask))[sig_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                    vmin=0, vmax=len(sig_mask))

    ax_scatter.axhline(0, color="k", lw=2, zorder=0)
    ax_scatter.axvline(0.1, color="k", lw=2, zorder=0)

    limit_ratio = 1.2

    if gamma == 3:
        ax_scatter.set_xlim(-mean_assort.max()*limit_ratio + 0.2, mean_assort.max()*limit_ratio)
    else:
        limit_ratio = 1.4
        ax_scatter.set_xlim(-mean_assort.max()*limit_ratio + 0.2, mean_assort.max()*limit_ratio)
    ax_scatter.set_ylim(-np.abs(rr_sr - ss_sr).max()*limit_ratio, np.abs(rr_sr - ss_sr).max()*limit_ratio)

    ax_in = ax_scatter.inset_axes([0.05, 0.7, 0.3, 0.3], xlim=(-1e-3, 1e-3), ylim=(-1e-3, 1e-3))

    zero_mask = mean_assort < 1e-3
    np.random.seed(7)
    zero_x = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())
    zero_y = np.random.uniform(-8e-4, 8e-4, zero_mask.sum())

    ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="o", #marker="D",
                c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=0,
                vmin=0, vmax=len(sig_mask), alpha=(sig_mask[zero_mask] == False).astype(float))

    ax_in.scatter(zero_x, zero_y, s=scatter_size[zero_mask], marker="s", #marker="D",
                c=np.arange(len(sig_mask))[zero_mask]+1, cmap=clust_cmap_net, edgecolors="k", lw=2,
                vmin=0, vmax=len(sig_mask), alpha=sig_mask[zero_mask].astype(float))

    ax_in.set_xticks([])
    ax_in.set_yticks([])
    ax_in.set_title("Assortativity = 0", fontsize=16)
    ax_in.spines[:].set_linewidth(2)
    ax_scatter.indicate_inset_zoom(ax_in, edgecolor="black", lw=2, zorder=0)
    ax_scatter.axis("off")

    toy_pos = [
        [0.56, 0.46, 0.1, 0.08],
        [0.53, 0.2, 0.1, 0.08],
        [0.53, 0.71, 0.1, 0.08],
        [0.39, 0.71, 0.1, 0.08],
        [0.39, 0.2, 0.1, 0.08],
        [0.368, 0.46, 0.1, 0.08],
    ]
    toy_titles = [
        "",
        "Receiving\nCore-Periphery",
        "Sending\nCore-Periphery",
        "Disassortative\nFocusing",
        "Disassortative\nBroadcasting",
        "",
    ]
    for toy_i, (toy_mat, pos) in enumerate(zip(all_sample_graphs, toy_pos)):
        ax_toy = fig.add_subplot(pos, facecolor="none")
        ax_toy.imshow((toy_mat > 0).astype(int), cmap=cmaps["div_rb"],
                    interpolation="none", vmax=1.3)
        ax_toy.set_title(toy_titles[toy_i], fontsize=18)
        ax_toy.set_xticks([])
        ax_toy.set_yticks([])
        ax_toy.axis("off")

    l1_pos = -0.21
    main_ax.scatter([l1_pos-0.02], [-1], marker="o", s=400, color="gray", edgecolor="k", lw=0)
    main_ax.text(l1_pos, -1.005, "Bidirectional", fontsize=16, ha="left", va="center")

    l2_pos = 0.02
    main_ax.scatter([l2_pos-0.02], [-1], marker="s", s=400, color="gray", edgecolor="k", lw=2)
    main_ax.text(l2_pos, -1.005, "Significantly Directed", fontsize=16, ha="left", va="center")

    cbar_pos = [[0.4, 0.11, 0.23, 0.01]]

    for pos in cbar_pos:
        plot.draw_cbar(fig, cmap=cmaps["div_rb"], ax_pos=pos, ticks=[0, 10],
                    labels=["Receiving", "Sending"], fontsize=16)

    # Drawing boxes
    box_lw = 2
    axe_boxes = fig.add_subplot([0.1, 0.1, 0.81, 0.81], facecolor="none")
    axe_boxes.axis("off")
    axe_boxes.fill([0.03, 0.362, 0.362, 0.03], [0.525, 0.525, 0.95, 0.95], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.03, 0.362, 0.362, 0.03], [0.04, 0.04, 0.465, 0.465], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.66, 0.99, 0.99, 0.66], [0.525, 0.525, 0.95, 0.95], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.fill([0.66, 0.99, 0.99, 0.66], [0.04, 0.04, 0.465, 0.465], color="none", edgecolor="k", lw=box_lw)
    axe_boxes.set_xlim(0, 1)
    axe_boxes.set_ylim(0, 1)

    main_ax.plot([-0.305, -0.24], [0.7, 0.74], color="k", lw=box_lw)
    main_ax.plot([0.315, 0.23], [0.7, 0.74], color="k", lw=box_lw)
    main_ax.plot([-0.305, -0.24], [-0.7, -0.74], color="k", lw=box_lw)
    main_ax.plot([0.315, 0.23], [-0.7, -0.74], color="k", lw=box_lw)

    main_ax.plot([0.315, 0.255], [0.24, 0.13], color="k", lw=box_lw)
    main_ax.plot([-0.305, -0.25], [0.24, 0.13], color="k", lw=box_lw)
    main_ax.plot([0.315, 0.255], [-0.21, -0.1], color="k", lw=box_lw)
    main_ax.plot([-0.305, -0.25], [-0.21, -0.1], color="k", lw=box_lw)

    if not fast:
        fig.savefig(op.join(path_to_figures, f"03-Showcase-BICOM_K{edge_clusters_mat.max()}{fig_suffix}.png"),
                    dpi=600, bbox_inches='tight')
        fig.savefig(op.join(path_to_figures, f"03-Showcase-BICOM_K{edge_clusters_mat.max()}{fig_suffix}.pdf"),
                    dpi=600, bbox_inches='tight', format="pdf")

## F-Tract Validation

### Preparation code

In [ ]:
maxdelay = 50
minmeas = 49
minimpl = 2

path_to_ftract = op.join(path_to_data, "brain", "F-Tract")
prob, delay_dict, ftract_labels = dload.get_ftract_data(
    path_to_ftract,
    scale=2, maxdelay=maxdelay,
    min_measurements=minmeas, min_implants=minimpl,
    atlas_year="2018",
    verbose=True
    )

selected_features = list(delay_dict.keys())

print("FTract Features are:")
for feat in selected_features:
    print(f" - {feat}")

In [ ]:
n_k_tested = 3
k_for_corr = best10_list + 2

n_perm = 4999
use_beta = False
use_spearman = True
use_median = True
use_abs = True

save_perm = True
path_to_saveperm = f"./results/BrainKmeans/scale{scale}/permutations"

path_to_saveperm = op.join(path_to_derivatives, "permutations", f"scale{scale}")

fname = f"permutations_scale{scale}_gamma{gamma}-F_meas{minmeas}_impl{minimpl}-{n_perm}Perm"
fname += "-Pearson"*(not use_spearman) + "-Mean"*(not use_median) + "-Abs"*(use_abs) + f"-K{len(k_for_corr)}"
fname += ".pkl"
print(f"Savename is {fname}")

if op.isfile(op.join(path_to_saveperm, fname)):
# if False:
    perm_dict = dload.load(op.join(path_to_saveperm, fname))

    all_corr_bi = perm_dict["all_corr_bi"]
    all_corr_rand = perm_dict["all_corr_rand"]
    best_bi = perm_dict["best_bi"]
    best_rand = perm_dict["best_rand"]
    edge_scatter = perm_dict["edge_scatter"]
    edge_scatter_rand = perm_dict["edge_scatter_rand"]
    all_best_k = perm_dict["all_best_k"]
    all_n_nans = perm_dict["all_n_nans"]
    all_best_perms = perm_dict["all_best_perms"]
    k_for_corr = perm_dict["k_for_corr"]
    selected_features = perm_dict["selected_features"]
else:
    all_corr_bi = np.zeros((len(selected_features), len(k_for_corr)))
    all_corr_rand = np.zeros((len(selected_features), len(k_for_corr), n_perm))

    best_bi = [0]*len(selected_features)
    best_rand = [0]*len(selected_features)

    edge_scatter = [0]*len(selected_features)
    edge_scatter_rand = [0]*len(selected_features)

    all_best_k = np.zeros((len(selected_features)), dtype=int)
    all_n_nans = np.zeros((len(selected_features), len(k_for_corr)))
    all_best_perms = np.zeros((len(selected_features), len(k_for_corr), np.sum(graph > 0)), dtype=int)

    parallel = Parallel(n_jobs=14, return_as='generator')
    counter = tqdm(total=len(selected_features)*len(k_for_corr)* (1 + n_perm))
    for d_i, feat in enumerate(selected_features):
        delay = delay_dict[feat]
        invdelay = np.divide(1, delay, out=np.zeros_like(delay), where=delay > 0)

        g_ratio = graph.copy()
        
        nzer_mask = np.logical_and(delay > 0, graph > 0)
        nzer_mask = np.logical_and(np.isnan(delay) == False, nzer_mask)
        nzer_mask = np.logical_and(np.isnan(delay.T) == False, nzer_mask)
        
        graph_ratio = np.divide(graph, graph + graph.T, out=np.zeros_like(graph, dtype=float), where=nzer_mask)
        f_ratio = np.divide(delay, delay + delay.T, out=np.zeros_like(graph_ratio), where=nzer_mask)
        
        edge_scatter[d_i] = (graph_ratio[nzer_mask], f_ratio[nzer_mask])

        # Making sure we compute the correlation only on non-zero edges and on the same edges for both datasets
        graph_ratio[~nzer_mask] = np.nan
        f_ratio[~nzer_mask] = np.nan

        if use_abs:
            maxcorr = 0
        else:
            maxcorr = -1

        feat_n_nans = []
        b_perm = []

        print("Permutations for feature:", feat)
        for i, k in enumerate(k_for_corr):
            labs = fcluster(Z, t=k, criterion='maxclust')

            corr, n_nans, ftract_ratio, ec_ratio = dgsp.get_bicom_ratio(
                graph, labs, f_ratio, graph_ratio, use_median=use_median,
                use_spearman=use_spearman, use_beta=use_beta, return_scatter=True
            )
            all_n_nans[d_i, i] = n_nans
            all_corr_bi[d_i, i] = corr

            if use_abs and (np.abs(corr) > np.abs(maxcorr)):
                maxcorr = corr
                best_bi[d_i] = (ec_ratio, ftract_ratio)
                all_best_k[d_i] = labs.max()
            
            if not use_abs and (corr > maxcorr):
                maxcorr = corr
                best_bi[d_i] = (ec_ratio, ftract_ratio)
                all_best_k[d_i] = labs.max()
            
            if use_abs:
                maxrand = 0
            else:
                maxrand = -1
            e_rand = []

            all_perm = [np.random.permutation(labs) for _ in range(n_perm)]

            res_gen = parallel(
                delayed(dgsp.get_bicom_ratio)(
                    graph, perm, f_ratio, graph_ratio, use_median=use_median,
                    use_spearman=use_spearman, use_beta=use_beta, return_scatter=True, corr_type=corr_type
                    ) for perm in all_perm)
            
            for j, (corr2, _, ft_perm_ratio, ec_perm_ratio) in enumerate(res_gen):
                while np.isnan(corr2):
                    all_perm[j] = np.random.permutation(labs)
                    corr2, _, ft_perm_ratio, ec_perm_ratio = dgsp.get_bicom_ratio(graph, all_perm[j], f_ratio, graph_ratio, use_median=use_median, use_spearman=use_spearman, return_scatter=True)
                
                all_corr_rand[d_i, i, j] = corr2

                if corr == maxcorr:
                    if use_abs and (np.abs(corr2) > np.abs(maxrand)):
                        maxrand = corr2
                        best_rand[d_i] = (ec_perm_ratio, ft_perm_ratio)
                        all_best_perms[d_i, i] = all_perm[j]
                    
                    if not use_abs and (corr2 > maxrand):
                        maxrand = corr2
                        best_rand[d_i] = (ec_perm_ratio, ft_perm_ratio)
                        all_best_perms[d_i, i] = all_perm[j]
                
                counter.update(1)

    counter.close()

    if save_perm:
        os.makedirs(path_to_saveperm, exist_ok=True)
        dload.save(
            op.join(path_to_saveperm, fname),
            {
                "all_corr_bi": all_corr_bi,
                "all_corr_rand": all_corr_rand,
                "best_bi": best_bi,
                "best_rand": best_rand,
                "edge_scatter": edge_scatter,
                "edge_scatter_rand": edge_scatter_rand,
                "all_best_k": all_best_k,
                "all_n_nans": all_n_nans,
                "all_best_perms": all_best_perms,
                "k_for_corr": k_for_corr,
                "selected_features": selected_features,
                })

In [ ]:
k_ids = np.array([np.where(k_for_corr == o_k + 2)[0][0] for o_k in ordered_maximas if o_k + 2 in k_for_corr])

if len(k_ids) > 10:
    k_ids = k_ids[:10]

In [ ]:
from statsmodels.stats.multitest import multipletests

plot_selection = [0, 1, 3, 6]

all_pvals = []
for ax_i, d_i in enumerate(plot_selection):
    print(f"Feature: {selected_features[d_i]}")
    for i, k in enumerate(k_ids):
        corr_bi = all_corr_bi[d_i]

        n_bigger = (np.abs(all_corr_rand[d_i, k]) >= np.abs(corr_bi[k])).sum()
        p_val = (n_bigger + 1)/(n_perm + 1)
        all_pvals.append(p_val)

isisg, p_corr, _, _ = multipletests(all_pvals, alpha=0.05, method="fdr_bh", is_sorted=False)

### Figure

In [ ]:
bonf_thresh = 0.05 / (4 * n_k_tested)
print(f"Bonferroni threshold: {bonf_thresh}")

cmap_plot = cmaps["cluster_palette_soft"]
col_4plots = cmap_plot.resampled(len(selected_features) + 1)

print(f"Min possible p-value: {1/(1+n_perm)}")

x_plot = np.linspace(0, 1, 100)
perc_plot = 5
plot_loo = True

plot_selection = [0, 1, 6, 3]
feat_titles = ["F-Tract Asymmetries\nOnset Delay", "Latency Start", "Peak Delay", "Amplitude"]

fig, axes = plt.subplots(nrows=2, ncols=len(plot_selection),
                         figsize=(6*len(plot_selection), 10), gridspec_kw={"hspace":0.3})

best_per_d = np.zeros(len(plot_selection))
old_best_per_d = 0
for ax_i, d_i in enumerate(plot_selection):
    corr_bi = all_corr_bi[d_i]
    color = col_4plots(d_i + 1)

    rand_del = all_corr_rand[d_i]
    rand_del = np.nan_to_num(rand_del, nan=0)

    corr_bi = np.nan_to_num(corr_bi, nan=0)
    best_per_d = np.argmax(np.abs(corr_bi))

    axes[0, ax_i].axis("off")
    gs = GridSpecFromSubplotSpec(2, 2, subplot_spec=axes[0, ax_i].get_subplotspec(),
                                 wspace=0, hspace=0, width_ratios=[5, 1], height_ratios=[1, 5])
    ax_bests = [fig.add_subplot(gs[i]) for i in range(4)]

    corr_label = f"$K={k_for_corr[best_per_d]}, \\rho={corr_bi[best_per_d]:.2f}$"
    ax_bests[2].scatter(best_bi[d_i][0], best_bi[d_i][1],
                        s=100, color=color, edgecolor="k",
                        label=corr_label,
                        zorder=4)
    
    # linear regression line
    loo_yvals, loo_corrs = plot.get_loo_curves(best_bi[d_i][0], best_bi[d_i][1], x_plot=x_plot)

    if plot_loo:
        reg = LinearRegression().fit(best_bi[d_i][0].reshape(-1, 1), best_bi[d_i][1].reshape(-1, 1))
        reg.score(best_bi[d_i][0].reshape(-1, 1), best_bi[d_i][1].reshape(-1, 1))

        y_vals = reg.intercept_[0] + reg.coef_[0]*x_plot
        ax_bests[2].plot(x_plot, loo_yvals.mean(axis=0), color=color, zorder=3, lw=2)
        ax_bests[2].fill_between(x_plot, loo_yvals.min(axis=0), loo_yvals.max(axis=0),
                                 color=color, alpha=0.3, zorder=2, edgecolor="none")
        
    e_corr = spearmanr(edge_scatter[d_i][0], edge_scatter[d_i][1])[0]
    ax_bests[2].scatter(edge_scatter[d_i][0], edge_scatter[d_i][1],
                        s=20, color="gray", edgecolor="none", alpha=0.2,
                          label=f"Edges, $\\rho={e_corr:.2f}$",  zorder=0)
    
    ax_bests[2].set_xlim(0.401, 0.599)
    ax_bests[2].set_ylim(0.401, 0.599)

    ax_bests[2].set_ylabel(feat_titles[ax_i], fontsize=16)
    ax_bests[2].set_xlabel(f"Connectome Asymmetry", fontsize=16)
    ax_bests[2].spines[["top", "right"]].set_visible(False)
    ax_bests[2].spines[:].set_linewidth(2)
    ax_bests[2].tick_params(labelsize=14, width=2)
    ax_bests[2].legend(fontsize=14, loc='upper left')

    ax_bests[0].set_xlim(ax_bests[2].get_xlim())
    ax_bests[3].set_ylim(ax_bests[2].get_ylim())

    xkde = np.linspace(-0.1, 1.1, 1000)
    plot.plot_kde(edge_scatter[d_i][0], data_range=xkde, ax=ax_bests[0],
                  bw_adjust=0.1, fill=True, color="silver", alpha=0.8)
    plot.plot_kde(edge_scatter[d_i][1], data_range=xkde, ax=ax_bests[3],
                  bw_adjust=0.1, fill=True, color="silver", alpha=0.8, vertical=True)
    
    plot.plot_kde(best_bi[d_i][0], data_range=xkde, ax=ax_bests[0],
                  bw_adjust=0.2, fill=True, color=color, alpha=1)
    plot.plot_kde(best_bi[d_i][1], data_range=xkde, ax=ax_bests[3],
                  bw_adjust=0.2, fill=True, color=color, alpha=1, vertical=True)

    ax_bests[1].axis("off")
    ax_bests[0].axis("off")
    ax_bests[3].axis("off")

    minp = 1
    for i, k in enumerate(k_ids):
        rand_x = np.random.uniform(-0.4, 0.4, size=len(all_corr_rand[d_i, k]))
        axes[1, ax_i].scatter(
            rand_x + i, all_corr_rand[d_i, k], s=20,
            color="gray", alpha=0.1, edgecolors="none")
        
        axes[1, ax_i].bar(i, corr_bi[k], zorder=1,
                 lw=2, edgecolor="none", color=color, alpha=0.5)

        if best_per_d == k:
            axes[1, ax_i].bar(i, corr_bi[k], zorder=1,
                    lw=3, edgecolor="k", color="none")
        else:            
            axes[1, ax_i].bar(i, corr_bi[k], zorder=1,
                        lw=3, edgecolor=color, color="none")

        n_bigger = (np.abs(all_corr_rand[d_i, k]) >= np.abs(corr_bi[k])).sum()
        p_val = (n_bigger + 1)/(n_perm + 1)

        if p_val < 0.05:
            maxval = np.max(all_corr_rand[d_i])
            maxval = 0.75

            if p_val < isisg[ax_i * n_k_tested + i]:
                axes[1, ax_i].text(i, maxval, "$**$", ha="center", va="center", fontsize=20)
            else:
                axes[1, ax_i].text(i, maxval, "$*$", ha="center", va="center", fontsize=20)
        
        if p_val < minp:
            minp = p_val
    print(f"Min p-val for {feat}: {minp} (Bonferroni: {bonf_thresh})")

    axes[1, ax_i].axhline(0, color="k", lw=3, zorder=3)
    axes[1, ax_i].set_xticks(range(len(k_ids)), labels=[str(k_for_corr[k]) for k in k_ids])
    axes[1, ax_i].set_xlabel("Number of Clusters (K)", fontsize=16)
    axes[1, ax_i].spines[["top", "right"]].set_visible(False)
    axes[1, ax_i].spines[:].set_linewidth(2)
    axes[1, ax_i].tick_params(labelsize=14, width=2)
    axes[1, ax_i].set_ylim(-0.89, 0.89)

axes[1, 0].set_ylabel("Spearman Correlation", fontsize=16)

axes[0, 0].text(-0.18, 0.95, "A.", transform=axes[0, 0].transAxes, fontsize=20, fontweight='bold', va='center', ha='center')
axes[1, 0].text(-0.18, 1.05, "B.", transform=axes[1, 0].transAxes, fontsize=20, fontweight='bold', va='center', ha='center')

fig.savefig(op.join(path_to_figures, f"04-Scatter-Ftract{minmeas}_{minimpl}-{n_perm}perm-Top{n_k_tested}.png"), dpi=300, bbox_inches='tight')
fig.savefig(op.join(path_to_figures, f"04-Scatter-Ftract{minmeas}_{minimpl}-{n_perm}perm-Top{n_k_tested}.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## Summary Per Bundle

### Preparation code

In [ ]:
n_yeo = 7
# n_yeo = 17

path_to_yeo_corr = op.join(path_to_derivatives, "atlas_correspondence", "YeoNetworks")
yeo_fname = f"Laus2018_Yeo{n_yeo}-scale{scale}OneThal.pkl"

yeo_corr = dload.load(os.path.join(path_to_yeo_corr, yeo_fname))

corr_mat = yeo_corr["dice"]
yeo_labels = yeo_corr["labels"]

# Abrev. of networks
yeo_labels[2] = "Dorsal Attn."
yeo_labels[3] = "Ventral Attn."

s_e_labels = all_e_labels[2]
s_e_mat = np.zeros_like(edge_clusters_mat)
s_e_mat[edge_clusters_mat > 0] = s_e_labels

send, rec = dgsp.get_node_clusters(s_e_labels, s_e_mat, scale=True)

In [ ]:
e_lab_one_hot = np.array([s_e_labels == k+1 for k in range(s_e_labels.max())]).astype(float)
e_lab_one_hot = e_lab_one_hot / e_lab_one_hot.sum(axis=1, keepdims=True)

e2b = edge_to_bundle_dist / edge_to_bundle_dist.sum(axis=1, keepdims=True)
e2b_max = e2b.max(axis=0)

bicom_to_bundle = e_lab_one_hot @ e2b
b2b_max = bicom_to_bundle.max(axis=0)

sort_by_overlap = np.argsort(b2b_max)[::-1]

# uncorr_p = (1 + (e2b >= b2b_max).sum(axis=0)) / (1 + e2b.shape[0])
# from statsmodels.stats.multitest import multipletests
# isisg, p_corr, _, _ = multipletests(uncorr_p, alpha=0.05, method="fdr_bh", is_sorted=False)
# print(isisg)
# print(p_corr)

### Figure

In [ ]:
from string import ascii_uppercase

path_to_centroids = op.join(path_to_bundles, "atlas", "pop_average")

label_space = 1.2
width_mod = 4
maxoverlap = 3
surf_overlay = 0.1
node_alpha = True

sorted_sig_df = sig_df.sort_values(by='abs_dir', ascending=False)
sort_by_dir = [ind for ind, row in sorted_sig_df.iterrows() if row['is_sig']]

if gamma == 3:
    bicom_list = [9, 24, 1, 16, 0, 12, 11, 23]
    views = ["right", "left", "right", "left", "right", "left", "right", "left"]
    n_abundle = 12
else:
    bicom_list = [11, 29, 4, 26, 13, 33, 2, 12]
    views = ["left", "right", "left", "right", "left", "right", "left", "left"]
    n_abundle = 14

maxoverlaps = [maxoverlap]*len(bicom_list)

manual_ab_names = {
    'AC': "Anterior Commissure",
    'AF': "Arcuate Fasciculus",
    'CC_Oc': "Corpus Callosum Occipital",
    'CC_Te': "Corpus Callosum Temporal",
    'CC_Pa': "Corpus Callosum Parietal",
    'CC_Fr_1': "Corpus Callosum Front. 1",
    'CC_Fr_2': "Corpus Callosum Front. 2",
    'CC_Pr_Po': "Corpus Callosum Post.",
    'CG': "Cingulum",
    'CG_An': "Cingulum Ant.",
    'CG_Po': "Cingulum Post.",
    'CG_curve': "Cingulum Curve",
    'FAT': "Front. Aslant Tract",
    'FPT': "Fronto-pontine tract",
    'FPT_Brainstem': "Fronto-pontine & Brainstem",
    'FX': "Fornix",
    'ILF': "Inf. Long. Fasciculus",
    'MdLF': "Middle Long. Fasciculus",
    'OR_ML': "Optic Radiation",
    'PYT': "Pyramidal tract",
    'PYT_Brainstem': "Pyramidal & Brainstem",
    'SLF': "Sup. Long. Fasciculus",
    'UF': "Uncinate Fasciculus"
    }

nrows = len(bicom_list) // 2
fig, axes = plt.subplots(ncols=2, nrows=nrows, figsize=(20, 3 * nrows),
                         gridspec_kw={'wspace':0.02, 'hspace':0})
axes = axes.flatten()

abundle_cmap = cluster_cmap.resampled(n_abundle + 1)
all_ab_names = {}
for ax_i, bic_id in enumerate(bicom_list):
    best_ab = np.flip(np.argsort(bicom_to_bundle[bic_id]))

    last_overlap = 0
    n_overlap = 0
    ab_names = []
    for ab_i, ab in enumerate(best_ab):
        ab_name = a_bundles_labels.loc[ab, "BundleName"]
        
        if (ax_i in [0, 1]) or (ab_name.split("_")[0] != "AF"):
            if last_overlap == 0:
                last_overlap = bicom_to_bundle[bic_id, ab]

            if (bicom_to_bundle[bic_id, ab] < 0.8 * last_overlap) or (bicom_to_bundle[bic_id, ab] < 0.5 * bicom_to_bundle[:, ab].max()) or (n_overlap >= maxoverlaps[ax_i]):
                break
            else:
                last_overlap = bicom_to_bundle[bic_id, ab]
                ab_names.append(ab_name)
                n_overlap += 1

                ab_name_nolat = ab_name.replace("_L", "").replace("_R", "")
                if ab_name_nolat not in all_ab_names.keys():
                    all_ab_names.update({ab_name_nolat: len(all_ab_names.keys()) + 1})
    
    stripednames = [name.replace("_L", "").replace("_R", "") for name in ab_names]
    color_idxs = [all_ab_names[name] for name in stripednames]

    colors = np.array([abundle_cmap(idx) for idx in color_idxs])

    path_to_trk = [op.join(path_to_centroids, f"{name.replace(' ', '_')}.trk") for name in ab_names]

    e_clust_mask = np.zeros_like(edge_clusters_mat)
    e_clust_mask[edge_clusters_mat == bic_id + 1] = 1

    subnet_mask = corr_mat @ e_clust_mask @ corr_mat.T

    asymmetry = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'directionality'].values[0]

    if asymmetry < 0.5:
        subnet_mask = subnet_mask.T
        e_clust_mask = e_clust_mask.T

    is_sig = sig_df.loc[sig_df['Bicomm_ID'] == bic_id + 1, 'is_sig'].values[0]

    if not is_sig:
        print(f"Making bicom {bic_id + 1} bidirectional")
        e_clust_mask += e_clust_mask.T
        e_clust_mask[e_clust_mask > 0] = 1

        subnet_mask = (subnet_mask + subnet_mask.T)/2

    if len(ab_names) > 4:
        ab_name = ", ".join(ab_names[:4]) + "\n" + ", ".join(ab_names[4:]) 
    else:
        ab_name = " , ".join(ab_names)
    axes[ax_i].axis("off")
    
    gs = GridSpecFromSubplotSpec(1, 3, axes[ax_i].get_subplotspec(), width_ratios=[1, 1, 1.1], wspace=0)
    sub_axes = [fig.add_subplot(gs[i]) for i in range(3)]
    
    sub_axes[0].text(0, 0.9, f"{ascii_uppercase[ax_i]}.", fontsize=16, transform=sub_axes[0].transAxes)

    if not fast:
        # Tracks
        plot.plot_trk(path_to_trk, cmap=cmaps["div_rb"], upsample=None, n_slines=500,
                        brain_opacity=surf_overlay, overlay_slines=surf_overlay == 1,
                        trk_color_list=colors,
                        view=views[ax_i], linewidth=0.3, slines_alpha=1, axes=sub_axes[0])

        # Bicom
        axpos = sub_axes[1].get_position()
        cbar_pos = np.array(axpos.bounds)
        cbar_pos[0] += cbar_pos[2] * 0.1
        cbar_pos[2] *= 0.8
        cbar_pos[1] += cbar_pos[3] * 0.2
        cbar_pos[3] = 0.01
        plot.plot_bicom_tracts(1, e_clust_mask, labels, scale=2, cmap=cmaps["div_rb"], n_centroids=5,
                            brain_opacity=surf_overlay, overlay_slines=surf_overlay == 1,
                            linewidth=0.3, slines_alpha=1 - 0.01 * is_sig, bidir_col=(0.6, 0.6, 0.6, 1),
                            view=views[ax_i], axes=sub_axes[1], upsample=upsample,
                            fig=fig, plot_cbar=False, cbar_pos=cbar_pos, cbar_fontsize=14)
    # Networks
    sub_axes[2].axis("off")
    if views[ax_i] == "left":
        node_pos = node_centers[:, [1, 2]] * [-1, 1]
    else:
        node_pos = node_centers[:, [1, 2]]
    
    sub_axes[2] = plot.plot_yeo_summary(subnet_mask, yeo_labels, cmap=cmaps["div_rb_silver"], axes=sub_axes[2],
                                        width_mod=1, r_lab_only=True, manual_arrows=True, net_alpha=node_alpha)

    if ax_i < 2:
        sub_axes[0].set_title("Anatomical Bundles", fontsize=14)
        sub_axes[1].set_title("Bicommunity Streamlines", fontsize=14)
        sub_axes[2].set_title("Network Representation", fontsize=14)

if gamma == 3:
    ax_legend = fig.add_subplot([0.12, 0.04, 0.78, 0.5], facecolor='none')
else:
    ax_legend = fig.add_subplot([0.12, 0.02, 0.78, 0.5], facecolor='none')
ax_legend.axis("off")

label_names = [manual_ab_names[name] for name, num in all_ab_names.items()]
legend_handles = [
    Line2D([0], [0], color=abundle_cmap(num), lw=4, label=f"{label_names[ai]}") for ai, (name, num) in enumerate(all_ab_names.items())
]
ax_legend.legend(handles=legend_handles, fontsize=12, loc='lower left', ncols=4, title="Anatomical Bundles", title_fontsize=16)

cb = plot.draw_cbar(fig, cmap=cmaps["div_rb"].reversed(), ax_pos=[0.65, 0.05, 0.22, 0.04], fontsize=12,
                    ticks=[0, 10], labels=["Sending", "Receiving"])
cb.set_label("Bicommunity Streamlines and Networks", fontsize=16, labelpad=-75)

if not fast:
    overlay_suffix = f"-overlay{surf_overlay}" if surf_overlay < 1 else ""
    alpha_suffix = f"-NAlpha" if node_alpha else ""
    fig.savefig(op.join(path_to_figures, f"05-Bundle_summary-K{e_clust.max()}-Bicom{bicom_list}-max{maxoverlap}{overlay_suffix}{alpha_suffix}.png"), dpi=300, bbox_inches='tight')
    fig.savefig(op.join(path_to_figures, f"05-Bundle_summary-K{e_clust.max()}-Bicom{bicom_list}-max{maxoverlap}{overlay_suffix}{alpha_suffix}.pdf"), dpi=300, bbox_inches='tight', format="pdf")

## Centroids Visualizations

In [ ]:
sel_id = 1
selected_bicom = sort_by_p[sel_id] + 1
print(f"Bicom {selected_bicom} selected (rank {sel_id+1})")

fig, axes = plt.subplots(figsize=(10, 10))

plot.plot_bicom_tracts(
    selected_bicom,
    edge_clusters_mat,
    labels,
    scale,
    cmap=cmaps["div_rb"],
    view="ortho",
    linewidth=0.2,
    slines_alpha=0.99,
    n_centroids=5,
    axes=axes,
    fig=fig,
    plot_cbar=True,)

In [ ]:
sel_list = [3, 13]

fig, axes = plt.subplots(ncols=len(sel_list), figsize=(8*len(sel_list), 8))

for i, sel in enumerate(sel_list):

    asymm = sig_df.loc[sel-1, "directionality"]
    axes[i].text(100, 100, f"Bicom {sel} (asym: {asymm:1.2f})", fontsize=20, ha="center")

    e_mask = np.zeros_like(edge_clusters_mat)
    if asymm < 0.5:
        e_mask[edge_clusters_mat.T == sel] = sel
    else:
        e_mask[edge_clusters_mat == sel] = sel

    plot.plot_bicom_tracts(
        sel,
        e_mask,
        labels,
        scale,
        cmap=cmaps["div_rb"],
        view="back",
        axes=axes[i],
        fig=fig,
        n_centroids=5,
        plot_cbar=False,
        overlay_slines=False,
        brain_opacity=0.1,
        slines_alpha=0.99,
        linewidth=0.2,
        upsample=upsample
        )